# Counting: one to two

50 QA pairs, one vector per layer. Choose CAA or RepE in step 6.


## 1. Set up Colab

Use a fresh GPU runtime. Restart after installation, then run step 2.


In [ ]:
!nvidia-smi
!git clone --branch experiment/custom-steering-vectors --single-branch https://github.com/Nicerova7/show-o2-space.git /content/show-o2-space
!git clone https://github.com/showlab/Show-o.git /content/Show-o
!git -C /content/Show-o checkout 45a5a2de01d1ebd10cd5864d29310a76476cdf23
%pip install -q transformers==4.47.0 diffusers==0.31.0 huggingface-hub==0.27.1 accelerate==1.2.1 timm==1.0.12 einops==0.8.0 torchdiffeq==0.2.5 "numpy>=1.26,<3" "Pillow>=10.4,<12" sentencepiece==0.2.1

## 2. Load Show-o2

Run once after restarting.


In [ ]:
import json
import os
import shutil
import sys
from pathlib import Path

import torch
from google.colab import files
from IPython.display import Markdown, display
from PIL import Image

PROJECT = Path("/content/show-o2-space")
UPSTREAM = Path("/content/Show-o")
os.chdir(PROJECT)
sys.path[:0] = [str(PROJECT), str(UPSTREAM / "show-o2")]

from show_o2_space.adapter import MODEL_ID, ShowO2Adapter
from experiments.custom_utils import (
    prepare_pair, capture_pair, build_steering_vector,
    count_denoising_evaluations, steer_denoising, repe_direction,
)

SEED = 42
torch.manual_seed(SEED)
adapter = ShowO2Adapter()
OUTPUT = PROJECT / "outputs/counting/one_to_two"
OUTPUT.mkdir(parents=True, exist_ok=True)
print(MODEL_ID, torch.cuda.get_device_name(0), adapter.dtype)

## 3. Load the 50 pairs

Positive: two. Negative: one. Answers are supplied.


In [ ]:
DATA = PROJECT / "inputs/counting/one_to_two/build_pairs.json"
pairs = json.loads(DATA.read_text(encoding="utf-8"))
layers = list(range(len(adapter.layers)))
print("Pairs:", len(pairs), "| Layers:", len(layers))
print("POSITIVE:\n", pairs[0]["positive"])
print("\nNEGATIVE:\n", pairs[0]["negative"])

## 4. Inspect final tokens

No padding or EOS. Capture the final token of each answer.


In [ ]:
token_audit = []
for example in pairs:
    pair = prepare_pair(
        adapter, example["positive"], example["negative"], alignment="none",
    )
    for side, ids in pair["token_ids"].items():
        token_audit.append(dict(
            id=example["id"], side=side, position=len(ids) - 1,
            token_id=ids[-1], token=adapter.tokenizer.decode([ids[-1]]),
        ))

display(Markdown(
    "| Pair | Side | Position | Token ID | Final token |\n"
    "| --- | --- | --- | --- | --- |\n"
    + "".join(
        f"| {r['id']} | {r['side']} | {r['position']} | {r['token_id']} | {r['token']!r} |\n"
        for r in token_audit[:10]
    )
))

## 5. Extract hidden states

Last-token block outputs, all layers, stored on CPU in FP32.


In [ ]:
positive_states = {layer: [] for layer in layers}
negative_states = {layer: [] for layer in layers}

for index, example in enumerate(pairs, 1):
    pair = prepare_pair(
        adapter, example["positive"], example["negative"], alignment="none",
    )
    captured = capture_pair(adapter, pair, layers=layers)
    pair_result = build_steering_vector(
        captured, tokens="last", token_reduction="mean",
        direction_method="difference", layer_reduction="none",
    )
    for layer in layers:
        positive_states[layer].append(pair_result["positive"][layer][0])
        negative_states[layer].append(pair_result["negative"][layer][0])
    print(f"{index}/{len(pairs)}: {example['id']}")

h_two = {layer: torch.stack(positive_states[layer]) for layer in layers}
h_one = {layer: torch.stack(negative_states[layer]) for layer in layers}
differences = {layer: h_two[layer] - h_one[layer] for layer in layers}
del captured, pair_result, positive_states, negative_states
print("Per-layer states:", tuple(h_two[layers[0]].shape))

## 6. Choose CAA or RepE

Run only one of the next two code cells. Each replaces `vectors`; neither mixes layers.

### CAA
Mean of the 50 pair differences, per layer.


In [ ]:
method = "caa"
normalization = "none"
vectors = {layer: differences[layer].mean(dim=0) for layer in layers}

for layer in layers:
    print(layer, tuple(vectors[layer].shape), "norm:", vectors[layer].norm().item())


### RepE
First principal component of the centered pair differences, per layer; unit norm.


In [ ]:
method = "repe"
normalization = "unit"
vectors = {layer: repe_direction(differences[layer]) for layer in layers}

for layer in layers:
    print(layer, tuple(vectors[layer].shape), "norm:", vectors[layer].norm().item())


## 7. Inspect the selected vectors

Six coordinates from one layer.


In [ ]:
preview_layer = layers[0]
rows = zip(
    h_two[preview_layer][0, :6].tolist(),
    h_one[preview_layer][0, :6].tolist(),
    differences[preview_layer][0, :6].tolist(),
)
display(Markdown(
    "| Coordinate | First h_two | First h_one | Difference |\n"
    "| --- | --- | --- | --- |\n"
    + "".join(f"| {i} | {p:.6f} | {n:.6f} | {d:.6f} |\n"
              for i, (p, n, d) in enumerate(rows))
))
print("Method:", method, "| Layer:", preview_layer)
display(Markdown(
    "| Coordinate | Selected vector |\n| --- | --- |\n"
    + "".join(f"| {i} | {v:.6f} |\n"
              for i, v in enumerate(vectors[preview_layer][:6].tolist()))
))


## 8. Save vectors

Results are saved separately under `caa/` or `repe/`.


In [ ]:
RUN_OUTPUT = OUTPUT / method
RUN_OUTPUT.mkdir(parents=True, exist_ok=True)

metadata = dict(
    model=MODEL_ID, concept="counting", direction="one_to_two",
    n_pairs=len(pairs), alignment="none", tokens="last", layers=layers,
    method=method, normalization=normalization,
    capture="decoder block output before final norm",
    seed=SEED, dtype=str(adapter.dtype),
    project_revision=get_ipython().getoutput("git -C /content/show-o2-space rev-parse HEAD")[0],
    upstream_revision=get_ipython().getoutput("git -C /content/Show-o rev-parse HEAD")[0],
)
torch.save(dict(vectors=vectors, metadata=metadata), RUN_OUTPUT / "steering_vectors.pt")
torch.save(
    dict(positive=h_two, negative=h_one, differences=differences),
    RUN_OUTPUT / "last_token_states.pt",
)
(RUN_OUTPUT / "build_pairs.json").write_text(json.dumps(pairs, indent=2), encoding="utf-8")
(RUN_OUTPUT / "token_audit.json").write_text(json.dumps(token_audit, indent=2), encoding="utf-8")
(RUN_OUTPUT / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved:", RUN_OUTPUT)

## 9. Download build results


In [ ]:
archive = shutil.make_archive(f"/content/counting-build-{method}", "zip", RUN_OUTPUT)
files.download(archive)

## 10. Generate baseline

Same prompt, seed, and sampling settings for both images. Count real denoising evaluations.


In [ ]:
prompt = "One circle on a white background."
generation_kwargs = dict(steps=50, guidance=5.0, seed=SEED)

with count_denoising_evaluations(adapter) as baseline_trace:
    baseline = adapter.generate(prompt, **generation_kwargs)

total_evaluations = baseline_trace["evaluations"]
print("Measured evaluations:", total_evaluations)

## 11. Apply steering

Each selected layer uses its own vector. Edit the rules below.


In [ ]:
application_layers = "all"
iterations = "all"
text_rule = None
visual_rule = dict(tokens="all", operation="add", alpha=0.1)

with steer_denoising(
    adapter, vectors=vectors, total_evaluations=total_evaluations,
    layers=application_layers, iterations=iterations,
    text=text_rule, visual=visual_rule,
) as steering_trace:
    steered = adapter.generate(prompt, **generation_kwargs)


### Inspect hidden states (optional)


In [ ]:
preview_key = next(iter(steering_trace["preview"]))
preview = steering_trace["preview"][preview_key]
rows = zip(
    preview["before"].tolist(), preview["after"].tolist(),
    (preview["after"] - preview["before"]).tolist(),
)
print("Layer:part:", preview_key, "| iteration:", preview["iteration"], "| position:", preview["position"])
display(Markdown(
    "| Coordinate | Before | After | Change |\n| --- | --- | --- | --- |\n"
    + "".join(f"| {i} | {b:.6f} | {a:.6f} | {d:.6f} |\n"
              for i, (b, a, d) in enumerate(rows))
))


### Compare images

Left: baseline. Right: steering.


In [ ]:
width, height = baseline.size
comparison = Image.new("RGB", (width * 2, height))
comparison.paste(baseline, (0, 0))
comparison.paste(steered, (width, 0))
print("Left: baseline | Right: steered")
print("Evaluations:", steering_trace["evaluations"])
print("Layer/modality injections:", len(steering_trace["injections"]))
display(comparison)

## 12. Save and download images


In [ ]:
baseline.save(RUN_OUTPUT / "baseline.png")
steered.save(RUN_OUTPUT / "steered.png")
comparison.save(RUN_OUTPUT / "comparison.png")
torch.save(
    dict(
        prompt=prompt, generation=generation_kwargs, layers=application_layers,
        iterations=iterations, text=text_rule, visual=visual_rule,
        trace=steering_trace, vector_metadata=metadata,
    ),
    RUN_OUTPUT / "application.pt",
)
archive = shutil.make_archive(f"/content/counting-application-{method}", "zip", RUN_OUTPUT)
files.download(archive)